# BIG DATA: Concurso facebook
### Importar las librerías

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows',     100)
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error


### Leer los datos
[`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

In [ ]:
dataset = pd.read_csv("../input/big-data-murcia-facebook-competicion/train.csv", index_col="user_id")

### Ver los datos

In [ ]:
dataset #para ver nuestros datos

In [ ]:
dataset.shape # para ver cuantas filas y columnas tenemos

In [ ]:
dataset.dtypes # para ver los tipos de dato en las columnas

In [ ]:
import missingno as msno
msno.matrix(dataset, color=(0.6,0.6,0), sparkline = False); #veo gráficamente cuantos valores NaN tengo

In [ ]:
#dataset = dataset.fillna(0) # para sustituir los valores null por 0 

In [ ]:
#dataset = dataset.dropna() # para borrar todas las filas que contienen algún null 

In [ ]:
# dataset = dataset.dropna(axis='share') # para borrar la columnas que contiene algún null

In [ ]:
dataset = dataset.fillna(dataset.mean()) # para sustituir los valores null por la media

In [ ]:
dataset.nunique() # para ver valores únicos dentro de las columnas

In [ ]:
dataset = pd.get_dummies(dataset, drop_first=True) # para cambiar datos de tipo nominal a enteros 

In [ ]:
dataset.dtypes # compruebo que los tipos de datos son todos númericos

# Creo las variables X e Y 

In [ ]:
X = dataset
X = X.drop(["Page total likes"], axis=1)

y = dataset["Page total likes"] 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()

In [ ]:
regressor

In [ ]:
from sklearn.metrics import make_scorer
RMSE = make_scorer(mean_squared_error, squared=False)

In [ ]:
from sklearn.model_selection import cross_val_score
CV_scores = cross_val_score(regressor, X, y, cv=5, scoring=RMSE)
print("CV scores:                       ",CV_scores)
print("Media del CV score:              ",np.mean(CV_scores))
print("Desviación estándar del CV score: ",np.std(CV_scores))

In [ ]:
test  = pd.read_csv("../input/big-data-murcia-facebook-competicion/test.csv",  index_col="user_id")

In [ ]:
test  = test.fillna(test.mean())
test  = pd.get_dummies(test, drop_first=True)

In [ ]:
regressor.fit(X_train, y_train)

In [ ]:
y_pred = regressor.predict(test)

In [ ]:
sample_submission = pd.read_csv("../input/big-data-murcia-facebook-competicion/sample_submission.csv")
sample_submission["Page total likes"] = y_pred
sample_submission.to_csv('submission.csv',index=False)

In [ ]:
sample_submission

In [ ]:
dataset['like'] = np.log(dataset['like']+1) 

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6)) 
dataset["like"].value_counts(normalize=True, bins=50).sort_index().plot.bar();

In [ ]:
dataset['share'] = np.log(dataset['share']+1) 

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6)) 
dataset["share"].value_counts(normalize=True, bins=50).sort_index().plot.bar();

In [ ]:
dataset['Total Interactions'] = np.log(dataset['like']+1) 

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6)) 
dataset["Total Interactions"].value_counts(normalize=True, bins=50).sort_index().plot.bar();

In [ ]:
dataset['Lifetime People who have liked your Page and engaged with your post'] = np.log(dataset['like']+1) 

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6)) 
dataset["Lifetime People who have liked your Page and engaged with your post"].value_counts(normalize=True, bins=50).sort_index().plot.bar();

In [ ]:
X_train = X_train.fillna(X_train.mean())
X_val   = X_val.fillna(X_val.mean())
X_test  = X_test.fillna(X_test.mean())

In [ ]:
msno.matrix(X_test, color=(0.9,0.1,0.4), sparkline = False);

In [ ]:
media_del_target = y_train.mean()
media_del_target

In [ ]:
n_val = len(y_val)

y_pred = pd.Series(media_del_target, index=range(n_val))

In [ ]:
from sklearn.metrics import mean_squared_error

RMSE = mean_squared_error(y_val, y_pred, squared=False)
print("RMSE",RMSE)

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()

regressor.fit(X_train, y_train)

In [ ]:
y_pred = regressor.predict(X_val)
RMSE = mean_squared_error(y_val, y_pred, squared=False)
print("RMSE",RMSE)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

regressor = DecisionTreeRegressor(max_depth=4, random_state=42)


regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_val)

RMSE = mean_squared_error(y_val, y_pred, squared=False)
print("RMSE",RMSE)


### Decision Tree (`max_depth=1`) = 7450.76
### Decision Tree (`max_depth=2`) = 4120.51
### Decision Tree (`max_depth=3`) = 1887.53
### Decision Tree (`max_depth=4`) = 1456.62

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(max_depth=5, random_state=42)


regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_val)

RMSE = mean_squared_error(y_val, y_pred, squared=False)
print("RMSE",RMSE)

### Random Forest Regressor (`max_depth=1`) = 6749.70
### Random Forest Regressor (`max_depth=2`) = 2634.22
### Random Forest Regressor (`max_depth=3`) = 1706.42
### Random Forest Regressor (`max_depth=4`) = 1366.57

In [ ]:
import eli5
eli5.show_weights(regressor, top=-1, feature_names = X_train.columns.tolist())

## XGBoosst Regressor

In [ ]:
import xgboost as xgb

regressor = xgb.XGBRegressor(eval_metric='rmsle')



regressor=xgb.XGBRegressor(learning_rate = 0.1,
                           n_estimators  = 100,
                           max_depth     = 2)

regressor.fit(X_train, y_train)

y_pred = regressor.predict(X_val)

RMSE = mean_squared_error(y_val, y_pred, squared=False)
print("RMSE", RMSE)

### XGBoost Regressor (`max_depth=1`) = 1431.35
### XGBoost Regressor (`max_depth=2`) = 1554.35
### XGBoost Regressor (`max_depth=3`) = 1636.51
### XGBoost Regressor (`max_depth=4`) = 1656.72

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(figsize=(16,10))
plot_importance(regressor, max_num_features=20, ax=ax)
plt.show();

In [ ]:
y_pred = regressor.predict(X_test)

RMSE = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE", RMSE)

# Para entregar mi modelo al concurso:

In [ ]:
test  = pd.read_csv("../input/big-data-murcia-facebook-competicion/test.csv",  index_col="user_id")

In [ ]:
# aqui haz a "test" lo que has hecho a "X_train"
test  = test.fillna(test.mean())
test  = pd.get_dummies(test, drop_first=True)

In [ ]:
test['like'] = np.log(test['like']+1) 
test['share'] = np.log(test['share']+1) 
test['Total Interactions'] = np.log(test['like']+1) 
test['Lifetime People who have liked your Page and engaged with your post'] = np.log(test['Lifetime People who have liked your Page and engaged with your post']+1)

In [ ]:
display(test)

In [ ]:
y_pred = regressor.predict(test)


In [ ]:
sample_submission = pd.read_csv("../input/big-data-murcia-facebook-competicion/sample_submission.csv")
sample_submission["Page total likes"] = y_pred
sample_submission.to_csv('submission.csv',index=False)